In [1]:
import subprocess
import librosa
import numpy as np
import tensorflow as tf
import os

def convertir_a_wav(ruta_entrada, sr=16000):
    """Convierte cualquier formato de audio a .wav 16khz usando ffmpeg."""
    ruta_salida = ruta_entrada.rsplit('.', 1)[0] + '_16k.wav'
    subprocess.run([
        'ffmpeg', '-y', '-i', ruta_entrada,
        '-ar', str(sr), '-ac', '1', ruta_salida
    ], capture_output=True)
    return ruta_salida

def extraer_features(ruta_wav, n_frames=5, frame_length=2048, sr=16000):
    """Extrae features de Fourier igual que el ETL de Alexis."""
    audio, _ = librosa.load(ruta_wav, sr=sr)
    if len(audio) < frame_length:
        raise ValueError(f'Audio demasiado corto: {len(audio)} muestras mínimo {frame_length}')

    total     = len(audio)
    seg_size  = total // n_frames
    features  = []
    for i in range(n_frames):
        center = i * seg_size + seg_size // 2
        start  = max(0, center - frame_length // 2)
        end    = start + frame_length
        if end > total:
            end   = total
            start = end - frame_length
        frame   = audio[start:end]
        stft    = librosa.stft(frame, n_fft=frame_length, hop_length=frame_length+1)
        features.append(np.abs(stft))
    return np.array(features)  # shape: (5, 1025, 1)

def predecir_audio(ruta_archivo, modelo, umbral=0.5):
    """Pipeline completo: convertir → extraer features → predecir."""
    nombre = os.path.basename(ruta_archivo)
    print(f'\nAnalizando: {nombre}')

    # Convertir a wav 16khz
    ruta_wav = convertir_a_wav(ruta_archivo)
    print(f'  Convertido a WAV 16khz ✓')

    # Extraer features
    features = extraer_features(ruta_wav)
    X        = features[np.newaxis, ...]  # shape: (1, 5, 1025, 1)
    print(f'  Features extraídas: {X.shape} ✓')

    # Predecir
    prob     = modelo.predict(X, verbose=0).flatten()[0]
    pred     = 'SPOOF (falso)' if prob >= umbral else 'BONAFIDE (real)'
    confianza = prob if prob >= umbral else 1 - prob

    print(f'  Probabilidad spoof : {prob:.4f}')
    print(f'  Predicción         : {pred}')
    print(f'  Confianza          : {confianza*100:.1f}%')

    return prob, pred

# --- Cargar modelo ---
modelo = tf.keras.models.load_model('../Metricas/ETL_LatinAmerica/modelo3_liliana.keras')
print('Modelo cargado ✓')

# --- Predecir todos los archivos en la carpeta ---
CARPETA = '/Users/liliana/Documents/GitHub/Reto_Inteligencia_Artificial_Liliana_Daniele_Alexis/test_voces'
FORMATOS = ('.wav', '.m4a', '.mp3', '.mp4', '.flac', '.ogg')

archivos = [f for f in os.listdir(CARPETA) if f.lower().endswith(FORMATOS) and '_16k' not in f]

print(f'Archivos encontrados: {len(archivos)}')
resultados = []
for archivo in sorted(archivos):
    ruta = os.path.join(CARPETA, archivo)
    prob, pred = predecir_audio(ruta, modelo)
    resultados.append({'archivo': archivo, 'prob_spoof': round(prob, 4), 'prediccion': pred})

# --- Resumen ---
import pandas as pd
print('\n=== RESUMEN ===')
print(pd.DataFrame(resultados).to_string(index=False))


Modelo cargado ✓
Archivos encontrados: 1

Analizando: Liliana_Voz.m4a
  Convertido a WAV 16khz ✓


/Users/liliana/venv312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  Features extraídas: (1, 5, 1025, 1) ✓
  Probabilidad spoof : 0.0206
  Predicción         : BONAFIDE (real)
  Confianza          : 97.9%

=== RESUMEN ===
        archivo  prob_spoof      prediccion
Liliana_Voz.m4a      0.0206 BONAFIDE (real)
